# FLUX.1 [schnell] on 8 GB Turing — exploration

**Author:** Dan Landayan  
**Hardware:** NVIDIA GeForce RTX 2070 (8 GB, compute 7.5)  
**Stack:** diffusers + torch 2.5 cu121, sequential CPU offload, fp16 compute

This notebook walks the actual generation loop the package exposes. It is not the place where logic lives — that's in `src/`. The notebook is for *thinking* about the model: where it shines, where 8 GB constrains us, and what I'd build on top next.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Allow `import src.*` when the notebook is opened from notebooks/.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config import InferenceConfig
from src.pipeline import FluxGenerator

## Why FLUX.1 [schnell] on a Turing GPU

Three reasons this specific pairing makes sense for a portfolio piece:

1. **License.** schnell is Apache 2.0. The dev variant ships under a non-commercial research license; for anything I might want to show in public, schnell removes the friction.
2. **Distillation.** schnell is timestep-distilled to 4 steps. On the dev variant a 768×1360 image is ~50 steps at minimum; on schnell it's 4. This is what makes 8 GB Turing viable at all.
3. **Dtype constraint is real.** Turing (compute 7.5) does not have native bf16 tensor cores. It *can* run bf16 — via an emulated path — but ~2× slower than native fp16. The pipeline therefore loads weights in bf16 (matches the on-disk checkpoint) but compute happens in fp16. This single decision dominates wall-clock time on this hardware.

## Cost of CPU offload

Sequential CPU offload is the only thing that makes FLUX (~12 B params) fit in 8 GB at all. It works by keeping every module on the CPU and uploading just the one being executed for each step in the forward pass.

The trade is brutal but predictable: you pay PCIe-bandwidth latency on every step (~16 GB/s effective on a Gen3 x16 slot), which on a 4-step generation at 1360-wide adds ~30 s to the wall-clock. Without offload, the pipeline simply OOMs at load time on this card.

On a 24 GB 4090 you'd drop the `enable_sequential_cpu_offload()` line and run 5–8× faster. That's the cost basis I use for the `SETUP.md` hybrid-workflow argument.

In [ ]:
# Run the smoke test as the gate. If this fails, don't continue.
import subprocess, sys
result = subprocess.run([sys.executable, str(repo_root / "scripts" / "smoke_test.py")])
assert result.returncode == 0, "smoke test failed — fix the pipeline before generating"

## Generation gallery — neuro-flavoured prompts

I'm coming from a neuroscience PhD, so the prompts I find most useful for showing model strengths are scientific-visualisation adjacent. FLUX is unusually good at structured technical imagery (microscopy, anatomical diagrams) compared to most open-weight models I've used.

In [ ]:
cfg = InferenceConfig()
gen = FluxGenerator(cfg)

PROMPTS = [
    "a fruit fly brain calcium imaging visualization, fluorescent neurons firing on a dark background, scientific publication quality",
    "a transmission electron micrograph of a synaptic cleft, labelled vesicles, monochrome, textbook illustration style",
    "a coronal section of mouse hippocampus stained with Nissl, brightfield microscopy, high detail",
    "isometric diagram of a chemical synapse, neurotransmitters crossing the cleft, modern infographic style",
]

for i, p in enumerate(PROMPTS):
    img, m = gen.generate_with_metrics(p)
    out = cfg.output_dir / f"gallery_{i:02d}.png"
    img.save(out)
    print(f"{i:02d}  {m['elapsed_s']:.1f}s  {m['peak_vram_gb']:.2f} GB  -> {out.name}")

## Seed sweep — variance under a fixed prompt

Generating the same prompt across multiple seeds is the cheapest way to feel a model's prior. For schnell this is especially informative because the distillation collapses the timestep dimension — most of the variance you see here is the model's view of "plausible interpretation of the prompt", not noise schedule drift.

In [ ]:
from dataclasses import replace
from PIL import Image

sweep_prompt = "a fruit fly brain calcium imaging visualization, fluorescent neurons on dark background"
seeds = [0, 7, 42, 123, 2024]
tiles = []
for s in seeds:
    sweep_cfg = replace(cfg, seed=s)
    img, m = gen.generate_with_metrics(sweep_prompt)
    tile = img.resize((cfg.width // 4, cfg.height // 4))
    tiles.append(tile)
    print(f"seed={s:5d}  elapsed={m['elapsed_s']:.1f}s")

# Side-by-side composite for at-a-glance comparison.
w, h = tiles[0].size
strip = Image.new("RGB", (w * len(tiles), h))
for i, t in enumerate(tiles):
    strip.paste(t, (i * w, 0))
strip.save(cfg.output_dir / "seed_sweep.png")
strip

## What I'd try next

Concrete extensions if I had another week and a 4090:

- **LoRA on neuro imagery.** Fine-tune a rank-16 LoRA on a small (~200 image) curated set of confocal calcium-imaging stills. The hypothesis is that schnell's prior already includes "fluorescent microscopy" semantically; a LoRA would tighten the *style* — colour palette, signal-to-noise feel — without retraining the whole transformer.
- **Prompt → CLIP-score eval.** Build a small harness that scores each generation against the prompt using a held-out CLIP variant (preferably one not used in FLUX training, e.g. SigLIP). Run the seed sweep above with N=32 per prompt and report mean/variance — a cheap way to quantify "does this prompt actually work" without human eval.
- **Compute-budget sweep.** Same prompt at 1, 2, 4, 8 steps × {512, 768, 1024} resolution. The schnell distillation paper claims 4 steps is near-optimal; verifying that empirically on this hardware is one afternoon and produces a Pareto chart that's directly useful for residency reviewers.
- **Negative-prompt ablation.** schnell technically supports `negative_prompt` despite being CFG-free; what does it actually do in the no-CFG regime? My guess is "nothing useful", but checking is a 30-minute experiment.